## Projekt Containerisierung
In diesem Projekt wird Docker eingesetzt, um die gesamte Infrastruktur reproduzierbar, portabel und einfach verwaltbar zu machen. Data-Engineering Workflows erfordern oft mehrere miteinander verbundene Systeme, etwa Spark für verteilte Datenverarbeitung, PostgreSQL als Datenbank und Streamlit für das Dashboard. Mit Docker lassen sich diese Komponenten als isolierte Container ausführen, ohne Konflikte zwischen Bibliotheken oder Umgebungen.
Der entscheidende Vorteil liegt darin, dass alle Projektmitglieder und auch Produktionssysteme mit exakt derselben Umgebung arbeiten, und das ganz unabhängig vom lokalen Betriebssystem oder installierten Tools.



### Docker Compose
Docker Compose dient als Orchestrierungswerkzeug, um mehrere Container gleichzeitig zu starten und zu verbinden. Statt jeden Dienst einzeln zu starten beschreibt die docker-compose.yml das gesamte System in einem einzigen YAML-File.
So entsteht eine kleine, reproduzierbare Datenplattform, in der Spark, PostgreSQL und das Streamlit-Dashboard automatisch miteinander kommunizieren können.

<br/>
Der Aufbau von *docker-compose.yml* sieht wie folgt aus:

```yaml
version: "3.8"

services:
  spark-master:
    image: spark:3.5.7-scala2.12-java17-python3-ubuntu
    container_name: spark-master
    environment:
      - SPARK_NO_DAEMONIZE=true         # keep process in foreground
      - SPARK_PUBLIC_DNS=localhost      # optional: nicer UI links
    user: root
    command: >
        /bin/bash -c "apt-get update && pip install --upgrade pip && pip install psycopg2-binary && /opt/spark/sbin/start-master.sh"

    ports:
      - "7077:7077"
      - "8080:8080"
    volumes:
      - .:/app
    networks: [spark-net]

  spark-worker-1:
    image: spark:3.5.7-scala2.12-java17-python3-ubuntu
    container_name: spark-worker-1
    depends_on:
      - spark-master
    environment:
      - SPARK_NO_DAEMONIZE=true
    user: root
    command: ["/opt/spark/sbin/start-worker.sh", "spark://spark-master:7077"]
    ports:
      - "8081:8081"
    volumes:
      - .:/app
    networks: [spark-net]

  db:
    image: postgres:16
    container_name: nyc_tlc_postgres16
    restart: unless-stopped
    environment:
      - POSTGRES_USER=appuser
      - POSTGRES_PASSWORD=group8
      - POSTGRES_DB=postgres
    ports:
      - "5432:5432"
    volumes:
      - pgdata:/var/lib/postgresql/data
      - ./init:/docker-entrypoint-initdb.d:ro
    networks: [spark-net]

  streamlit:
    build:
      context: .
      dockerfile: Dockerfile.streamlit
    container_name: streamlit-dashboard
    depends_on:
      - db
    ports:
      - "8501:8501"
    volumes:
      - .:/app
    networks: [spark-net]

volumes:
  pgdata:

networks:
  spark-net:
    driver: bridge

#### Übersicht der Docker Services


##### **Spark-Master**

Der Spark-Master ist das Steuerzentrum des Spark-Clusters. Er verwendet das Image *spark:3.5.7-scala2.12-java17-python3-ubuntu* und startet über das Kommando:

`/opt/spark/sbin/start-master.sh`

Dabei werden Umgebungsvariablen gesetzt, um die Ausführung im Vordergrund zu halten (SPARK_NO_DAEMONIZE=true) und eine lokale Oberfläche auf Port 8080 bereitzustellen. Das Volume-Mapping (.:/app) bindet den Projektordner in den Container ein, damit Spark auf die lokalen Skripte und Datendateien zugreifen kann.


##### **Spark-Worker-1**

Der Worker-Container führt die eigentlichen Rechenoperationen aus. Er verbindet sich über
`spark://spark-master:7077`
mit dem Master und führt die von ihm zugewiesenen Tasks aus.
Durch das gleiche Volume-Mapping kann er auf dieselben Skripte und Daten zugreifen. 

##### **PostgreSQL-Datenbank**
Die Datenbank `db` basiert auf dem offiziellen Image `postgres:16`. Sie dient als persistente SQL basierte Speicherstruktur für transformierte Daten.
Über die Umgebungsvariablen wird ein Benutzer (appuser) und ein Passwortdefiniert.
Das Volume `pgdata` sorgt dafür, dass die Datenbankdaten auch nach einem Neustart erhalten bleiben.
Die Initialisierungsdateien im Ordner ./init werden beim ersten Start automatisch ausgeführt, um Tabellen oder Schemas anzulegen.

##### **Streamlit**
Der Streamlit-Service bildet die Visualisierungsebene des Projekts. Er wird über ein eigenes *Dockerfile* gebaut, damit die Python-Abhängigkeiten gezielt installiert und die Projektdateien in den Container kopiert werden können.



### Streamlit Konfiguration
Während Spark und PostgreSQL über offizielle Images laufen, benötigt Streamlit eine auf das Projekt zugeschnittene Python-Umgebung, die nicht in einem Standardimage enthalten ist. Ein dediziertes Dockerfile dient dazu, eine eigenständige und reproduzierbare Laufzeitumgebung für das Streamlit-Dashboard zu schaffen. 
##### **Dockerfile von Streamlit** *(Dockerfile.streamlit)*



```yaml 
FROM python:3.10-slim

WORKDIR /app
COPY streamlit_app.py .
COPY requirements.txt .
COPY data_utilities/ ./data_utilities/
COPY dashboard_sections/ ./dashboard_sections/
COPY taxi_zones/ ./taxi_zones/

RUN pip install --upgrade pip
RUN pip install -r requirements.txt

EXPOSE 8501

CMD ["streamlit", "run", "streamlit_app.py", "--server.port=8501", "--server.address=0.0.0.0"]
```

### Importprozess für Spark
Anstatt die Rohdaten direkt in PostgreSQL zu importieren, nutzt das Projekt Spark, um große Datenmengen effizient zu verarbeiten und lädt die Daten nach erfolgreicher Transformation in die Datenbank.
Nach dem Starten des Docker Clusters muss somit der Datensatz über folgenden Docker Befehl in die Datenbank geladen werden:
```yaml
docker exec spark-master /opt/spark/bin/spark-submit \
  --master spark://spark-master:7077 \
  --driver-memory 2g \
  --conf "spark.executor.memory=2g" \
  --jars /app/postgresql-42.7.8.jar \
  /app/load_nyc_dataset.py \
  --data-file /app/fhvhv_tripdata_2025-07.parquet \
  --pg-host db --pg-port 5432 --pg-db postgres --pg-user appuser --pg-pass group8 \
  --partitions 16 --batchsize 1000
```
Dieser Befehl führt innerhalb des Spark-Master Containers das Python-Skript `load_nyc_dataset.py` aus. Das Skript liest den NYC TLC Datensatz ein, bereitet ihn mit Spark vor und lädt die Daten anschließend in die Datenbank.


